# Pre-processing

Module imports

In [1]:
import multiprocessing as mp
import os
import sys
from functools import partial
from itertools import chain

import numpy as np
import pandas as pd
import trimesh
from numba import njit
from scipy.spatial import KDTree
from tqdm import tqdm

# make multiprocessing compatible with macOS
# mp.set_start_method('fork', force=True)

Adding PATH 

In [2]:
# Go up THREE levels (project root directory)
project_root = os.path.dirname(os.path.dirname(os.getcwd()))
# Append the new path to sys.path
if project_root not in sys.path:
    sys.path.append(project_root)
    print("Project root added to sys.path")
else:
    print("Project root already in sys.path")

Project root added to sys.path


Parsing `.gro` file

In [ ]:
from utils import gro_processing as gp

# Data directory can be accessed due to root PATH we set previously
path = 'data/npt-HK4.gro'
file = os.path.join(project_root, path)

# Extracts data from .gro file into DataFrame (unsorted)
data, title, num_atoms, box_dimensions = gp.read_gro(file, multiply=10) # convert nm to Å

# Create multi-index DataFrame (sorted)
df_gro = gp.dataframe_gro(data, box_dimensions[0], positions=True, velocities=False, oxygen_midpoints=False)

# Dictionary of molecules {res_id: [(atom_name, np.array([x, y, z])), ...]}
molecules = {}

for res_id in df_gro.index.get_level_values('res_id').unique():
    residue_data = df_gro.xs(res_id, level='res_id')
    
    # Using itertuples() - much faster for large DataFrames
    atom_list = [(row.Index, np.array([row.x, row.y, row.z])) for row in residue_data.itertuples()]
    
    molecules[res_id] = atom_list
    
# molecules # display 

# Setup

Various configs and atom radii data for constructing molecule mesh

In [5]:
# --- CONFIG ---                 
sphere_radius_scale = 2.0                # balls (atoms): 2.0 × van der Waals radius   # idk why 1.5 :(
bond_radius = 0.1                        # sticks (bonds): cylinder radius in Å
sphere_subdiv = 2                        # atom sphere detail (2 is moderate)

# --- Radii (Å) ---
vdw = {"H":1.20,"C":1.70,"N":1.55,"O":1.52,"F":1.47,"P":1.80,"S":1.80,"Cl":1.75,"Na":2.27,"K":2.75,"Ca":2.31}
cov = {"H":0.31,"C":0.76,"N":0.71,"O":0.66,"F":0.57,"P":1.07,"S":1.05,"Cl":1.02,"Na":1.66,"K":2.03,"Ca":1.74}

# --- Element inference ---
hash_atom = {"OW": "O", "HW": "H", "HW1": "H", "HW2": "H"} # Atom name aliases
color_map = {'O': np.array([220, 20, 60, 255], dtype=np.uint8),     # Crimson Red
             'N': np.array([65, 105, 225, 255], dtype=np.uint8),    # Royal Blue
             'C': np.array([128, 128, 128, 255], dtype=np.uint8),   # Gray
             'H': np.array([230, 230, 230, 255], dtype=np.uint8),   # Light Gray
            }

def infer_element(atomname):
    # common water aliases
    if atomname in hash_atom: 
        return hash_atom[atomname]
    
    # simple: first letter, capitalize second if lowercase
    a = ''.join([c for c in atomname if c.isalpha()])   
    
    # join() joins items in an iterable into one string, '' is specified as the separator.
    # isalpha() method returns True if all the characters are alphabet letters (a-z).

    if a == '': 
        return "C"

    if len(a) >= 2 and a[1].islower(): 
        return (a[0]+a[1]).capitalize()
    
    return a[0].upper()

Helper function for MIC related calculations

In [6]:
# --- Minimum Image Convention (vector) ---
@njit(cache=True, fastmath=True)
def mic_vector(dx, box_dimensions):
    """Return minimum-image displacement for vector dx under PBC."""
    k = np.rint(dx / box_dimensions)
    return dx - k * box_dimensions

@njit(cache=True, fastmath=True)
def mic_distance(a, b, box_dimensions):
    """Return MIC distance between two 3D points a,b."""
    return np.linalg.norm(mic_vector(b - a, box_dimensions))

# Helper function to wrap points into the primary box
@njit(cache=True, fastmath=True)
def wrap_points(points, box_dimensions):
    """
    Wrap points into the primary simulation box using periodic boundary conditions.
    """
    return points % box_dimensions


# compile JIT functions
x = y = np.array([1.0, 2.0, 3.0])
_ = mic_vector(x, y) 
_ = mic_distance(x, x, y)
_ = wrap_points(x,y)

# Generating molecule meshes

This part has 3 function with the following uses:
- Building a single molecule mesh
- Looping through all 1501 molecules to draw mesh
- Saving the meshes into `.ply` file as binary

## Ball-and-stick molecule model

Function to build a single molecule mesh

In [7]:
# --- Build ball-and-stick with PBC ---
def build_molecule_ballstick(coords, elements, 
                             vdw, cov, box_length, 
                             sphere_radius_scale=0.3, 
                             sphere_subdiv=2, 
                             bond_radius=0.1):
    """
    Build trimesh ball-and-stick model under periodic boundary conditions.
    """

    # radii arrays
    vdw_r = np.array([vdw.get(e, 1.70) for e in elements])
    cov_r = np.array([cov.get(e, 0.76) for e in elements])
    
    # number of atoms
    n = len(coords)
    
    
    # --- Process spheres using chunking--- 
    # Base icosphere (unit radius)
    base_sphere = trimesh.creation.icosphere(subdivisions=sphere_subdiv, radius=1.0)
    # Precompute position, radii and number of vertices for each sphere
    coords_wrapped = np.mod(coords, box_length)
    scaled_radii = vdw_r * sphere_radius_scale  
    num_sphere_vertices = len(base_sphere.vertices)
    # Chunk (pre-allocate) array to hold icosphere object
    meshes_sphere = np.empty(n, dtype=object)
    
    for idx, (pos, r) in enumerate(zip(coords_wrapped, scaled_radii)):
        sphere = base_sphere.copy()
        sphere.apply_scale(r)
        sphere.apply_translation(pos)
        sphere_color = color_map.get(elements[idx], color_map.get('C')) # element not found default to grey (C)
        sphere.visual.vertex_colors = np.tile(sphere_color, (num_sphere_vertices, 1))
        meshes_sphere[idx] = sphere 
        
        
    # --- Process cylinder with numba --- 
    # For small molecules roughly < 1500 atoms, brute force is most efficient
    # Base cylinder (for number of faces and color assignment)
    base_cyl = trimesh.creation.cylinder(radius=1.0, height=1.0, sections=24)
    num_bond_faces = len(base_cyl.faces)
    bond_color = color_map.get('H') # light grey
    
    meshes_cylinder = []
    if n < 1500: # brute force
        for i in range(n):
            for j in range(i+1, n):
                d = mic_distance(coords[i], coords[j], box_length)
                thr = 1.2 * (cov_r[i] + cov_r[j])
                if d < thr:
                    # Unwrap j relative to i
                    disp = mic_vector(coords[j] - coords[i], box_length)
                    pos_i = np.mod(coords[i], box_length)
                    pos_j = pos_i + disp  # may fall outside box but correct bond vector
                    seg = np.vstack((pos_i, pos_j))
                    cyl = trimesh.creation.cylinder(radius=bond_radius,
                                                    segment=seg, sections=24)
                    cyl.visual.face_colors = np.tile(bond_color, (num_bond_faces, 1))
                    meshes_cylinder.append(cyl)
    else: # k-d tree 
        tree = KDTree(coords, leafsize=10)
        for i in range(n):
            _nearest_coords, nearest_index = tree.query(coords[i], k=9) # 8 neighbors
            for j in nearest_index[1:]: # Skip itself
                d = mic_distance(coords[i], coords[j], box_length)
                thr = 1.2 * (cov_r[i] + cov_r[j])
                if d < thr:
                    # Unwrap j relative to i
                    disp = mic_vector(coords[j] - coords[i], box_length)
                    pos_i = np.mod(coords[i], box_length)
                    pos_j = pos_i + disp # may fall outside box but correct bond vector
                    seg = np.vstack((pos_i, pos_j))
                    cyl = trimesh.creation.cylinder(radius=bond_radius,
                                                    segment=seg, sections=24)
                    cyl.visual.face_colors = np.tile(bond_color, (num_bond_faces, 1))
                    meshes_cylinder.append(cyl)

    # --- Merge all into one mesh ---
    molecule = trimesh.util.concatenate(meshes_sphere.tolist() + meshes_cylinder)
    return molecule

## Molecules to meshes

Function to parallelize processing all 1501 molecules.

In [8]:
# Helper function process a single molecule 
def process_single_molecule(mol_items,  
                            vdw, cov, box_length,
                            sphere_radius_scale=0.3, 
                            sphere_subdiv=2, 
                            bond_radius=0.1):
    """Process a single molecule and return (mol_id, mesh)"""
    mol_id, atoms = mol_items
    elements = [infer_element(name) for name, _ in atoms]
    coords = np.vstack([pos for _, pos in atoms])
    
    mesh = build_molecule_ballstick(
        coords, elements, vdw, cov, box_length,
        sphere_radius_scale=sphere_radius_scale,
        sphere_subdiv=sphere_subdiv,
        bond_radius=bond_radius
    )
    
    return mol_id, mesh

In [9]:
def molecules_to_meshes_parallel(molecules, 
                                 vdw, cov, box_dimensions,
                                 sphere_radius_scale=0.3,
                                 sphere_subdiv=2,
                                 bond_radius=0.1,
                                 num_processes=None):
    """
    Convert parsed molecules into trimesh meshes.

    Parameters
    ----------
    molecules : dict[int, list[tuple]]
        From parse_gro(): molecules[i] = [(atomname, coords), ...]
        coords must be in Å
    box_dimensions : np.ndarray
        Simulation box_dimensions (Å), shape (3,) for orthorhombic or (3,3) for triclinic
    vdw, cov : dict
        Van der Waals and covalent radii
    sphere_radius_scale : float
        Scaling factor for atom radii
    sphere_subdiv : int
        Subdivisions for icosphere (mesh resolution)
    bond_radius : float
        Cylinder radius for bonds
    num_processes: int/None
        Number of CPU cores to use (all if not specified)

    Returns
    -------
    mol_meshes : dict[int, trimesh.Trimesh]
        Dictionary of molecule meshes keyed by mol_id
    """
    # assume orthorhombic box for now
    if box_dimensions.shape == (3,):
        box_length = box_dimensions
    else:
        raise NotImplementedError("Triclinic box handling not yet implemented")
    
    
    # --- Multiprocessing ---
    # number of processes (use all CPUs if not specified)
    if num_processes is None:
        num_processes = mp.cpu_count()
    
    # partial function with fixed parameters
    process_func = partial(
        process_single_molecule,
        vdw=vdw,
        cov=cov,
        box_length=box_length,
        sphere_radius_scale=sphere_radius_scale,
        sphere_subdiv=sphere_subdiv,
        bond_radius=bond_radius
    )

    # parallel execution
    mol_items = list(molecules.items())
    num_mol = len(mol_items)
    
    with mp.Pool(processes=num_processes) as pool:
        tqdm_iterator = tqdm(
            pool.imap(process_func, mol_items),
            total=num_mol,
            desc=f'Processing {num_mol} molecules with {num_processes} logical cores',
            colour='#7BC8F6'
        )
        
        mol_meshes = list(tqdm_iterator) # initialise progress bar
        
    return dict(mol_meshes) # return dictionary

In [9]:
# Now mol_meshes is {1: Trimesh(...), 2: Trimesh(...), ..., 1501: Trimesh(...)}
# Takes ~ 50.0 seconds
mol_meshes = molecules_to_meshes_parallel(molecules, vdw, cov, box_dimensions, num_processes=None)
print(len(mol_meshes))        # 1501
print(mol_meshes[1])          # trimesh.Trimesh object

Processing 1501 molecules with 8 logical cores:  11%|█         | 163/1501 [00:05<00:42, 31.71it/s]Process ForkPoolWorker-5:
Process ForkPoolWorker-1:
Process ForkPoolWorker-6:
Process ForkPoolWorker-8:
Process ForkPoolWorker-2:
Process ForkPoolWorker-4:
Process ForkPoolWorker-3:
Traceback (most recent call last):

Traceback (most recent call last):


KeyboardInterrupt: 

## Export molecule meshes to `.ply` file

In [15]:
def export_single_mesh(args, directory, file_type):
    """Export a single mesh to a PLY file."""
    mol_id, mesh = args
    ply_file = os.path.join(directory, f'molecule_{mol_id:04d}.ply')
    mesh.export(ply_file, file_type)
    
    return None


def export_meshes_to_ply(mol_meshes, directory, file_type='ply', num_processes=None):
    """Export all meshes to PLY files in the specified directory."""
    # creates directory if it doesn't exist
    if not os.path.exists(directory): 
        os.makedirs(directory)
        
    # number of processes (use all CPUs if not specified)
    if num_processes is None: 
        num_processes = mp.cpu_count()
    
    # partial function with fixed parameters
    process_func = partial(
        export_single_mesh,
        directory=directory,
        file_type='ply'
    )
    
    # parallel execution
    total = len(mol_meshes)
    args = mol_meshes.items()
    with mp.Pool(processes=num_processes) as pool:
        tqdm_iterator = tqdm(
            # Use imap_unordered for a lazy, memory-efficient map
            pool.imap_unordered(process_func, args),
            total=total,
            desc=f'Exporting {total} meshes with {num_processes} logical cores',
            colour='#7BC8F6'
        )
        list(tqdm_iterator) # Initialise progress bar
        
    return None

In [ ]:
# Warning! For 1501 molecules, takes up ~ 1.0GB disk space
# Takes ~ 4.0 seconds
export_meshes_to_ply(mol_meshes, directory='./molecule_meshes', file_type='ply', num_processes=None)

# Import mol_meshes from `.ply` files

In [6]:
def count_ply(path): # just for progress bar
    """Count the total number of PLY files in a directory."""
    count = 0
    with os.scandir(path) as entries:
        for entry in entries:
            if entry.is_file() and entry.name.endswith('.ply'):
                count += 1
    return count


def find_ply(path):
    """Generator yielding the full path of all PLY files in a directory"""
    with os.scandir(path) as entries:
        for entry in entries:
            if entry.is_file() and entry.name.endswith('.ply'): 
                yield entry.path


def load_single_mesh(args):
    """
    Worker function to load a single mesh.
    This function is designed to be called by a multiprocessing pool.
    Note that, mol_id is inferred from the number of files.
    """
    mol_id, ply_file = args
    mesh = trimesh.load(ply_file, file_type='ply', process=True)
    return mol_id, mesh


def load_meshes_from_ply(directory, num_processes=None):
    """
    Load all PLY files from a directory into a dictionary of trimesh objects
    using multiprocessing.
    """    
    # Determine the number of processes
    if num_processes is None: 
        num_processes = mp.cpu_count()
    
    # Get the total number of files to show progress
    total_files = count_ply(directory)
    
    # Create the iterable of arguments for the workers
    args = enumerate(find_ply(directory))
    
    with mp.Pool(processes=num_processes) as pool:
        # Use imap_unordered for a lazy, memory-efficient map
        tqdm_iterator = tqdm(
            pool.imap(load_single_mesh, args),
            total=total_files,
            desc=f'Loading {total_files} meshes with {num_processes} cores',
            colour='#7BC8F6'
        )
        
        # Iterate over the results from the pool and populate the dictionary
        meshes = {mol_id + 1: mesh for mol_id, mesh in tqdm_iterator}
            
    return meshes

In [7]:
# Takes ~ 6.0 seconds
# Cleans and optimizes loaded meshes by trimming unused vertices and faces
mol_meshes = load_meshes_from_ply("./molecule_meshes", num_processes=None)
mol_meshes[1]

Loading 1501 meshes with 8 cores: 100%|██████████| 1501/1501 [00:21<00:00, 70.13it/s] 


<trimesh.Trimesh(vertices.shape=(18212, 3), faces.shape=(35904, 3))>

## Visualisation

In [8]:
# scene = trimesh.Scene(list(mol_meshes.values())[0:30])
# scene.show()

# Blocking Molecules

## Electron Clump Centre Centroids

User defines the region within molecule to be considered as _electron clump centre_. 

Then, for every molecule, the centroid of _electron clump cluster_ be calculated with respect to that region. These centroids are saved as an array in `e_centroids` with shape `(n,3)`.

### Manually define electron clump cluster

In [ ]:
# Helper function to extend user input
def extend_input(user_input):
    if '-' in user_input:
        start, end = user_input.split('-')
        return [*range(int(start), int(end) + 1)]
    else:
        return [int(user_input)]
    
# User selects atoms within a single molecule 
def user_indices_prompt(df_gro):
    # parameters
    num_atoms_per_mol = df_gro.loc[(1,),:].shape[0] # first residue
    num_res = df_gro.index.levels[0][-1]

    # user chooses atoms within molecule
    print("Insert atom id for one molecule.")
    print("Use the following format: '20-30; 30; 20; 40-60' (for range use '-', for multi-input split using ';' ")
    # user_input = input("Enter atom id:")
    user_input = "32-53" # for testing purposes
    
    print('-' * 40)
    print(f"You entered: [{user_input}]")

    # extract atom indices that user selected
    try: 
        indices = [extend_input(i) for i in user_input.split(";")]
        indices = np.unique(list(chain.from_iterable(indices)))
        print(f"Selected atom_id(s): {indices}") # Verify selection
    except: 
        print("Invalid input format. Please use the specified format.")
        
    # filters all molecules based on user selected atoms
    indices_all = [indices]
    for i in range(1, num_res):
        indices_all.append(indices + i * num_atoms_per_mol)
    
    return np.concatenate(indices_all) # flatten the list

In [26]:
# Extract atoms from all residue electron clump cluster 
indices_all = user_indices_prompt(df_gro) # atom_id of all electron clump cluster atoms
mask = df_gro["atom_id"].isin(indices_all)
df_e_clumps = df_gro.loc[mask].copy()

# display(df_e_clumps)

Insert atom id for one molecule.
Use the following format: '20-30; 30; 20; 40-60' (for range use '-', for multi-input split using ';' 
----------------------------------------
You entered: [32-53]
Selected atom_id(s): [32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47 48 49 50 51 52 53]


### E-Centroids Calculations

In [27]:
# parameters
num_res = df_gro.index.levels[0][-1]

def calculate_e_centroids(df_e_clumps, num_res, box_dimensions):
    """
    Calculate centroids of electron clump clusters for each molecule under PBC.

    Parameters
    ----------
    df_e_clumps : pd.DataFrame
        DataFrame containing selected atoms for electron clump clusters, indexed by (res_id, atom_name).
    num_res : int
        Number of molecules (residues).
    box_dimensions : np.ndarray
        Simulation box dimensions (nm).

    Returns
    -------
    df_e_centroids : pd.DataFrame
        DataFrame of shape (num_res, 3) with centroid coordinates for each molecule.
    """
    
    e_centroids = np.zeros((num_res, 3)) # chunking

    for res_id in range(1, num_res + 1):
        # selects residue (molecule) one-by-one
        res_atoms = df_e_clumps.loc[(res_id,slice(None)),['x','y','z']].to_numpy()
        ref_atom = res_atoms[0]
        rel_displacement = np.zeros_like(res_atoms) # chunking
        
        # rel displacement of all atoms from ref_atom
        for i in range(res_atoms.shape[0]):
            rel_displacement[i] = mic_vector(res_atoms[i] - ref_atom, box_dimensions)
        
        # calculate centroid of the molecule from origin [0,0,0]
        mean_displacement = np.mean(rel_displacement, axis=0)
        centroids_unwrapped = ref_atom + mean_displacement
        e_centroids[res_id-1] = wrap_points(centroids_unwrapped, box_dimensions)

    # convert to dataframe
    df_e_centroids = pd.DataFrame(e_centroids, columns=['x', 'y', 'z'], index=range(1, num_res + 1))
    
    return df_e_centroids

In [28]:
df_e_centroids = calculate_e_centroids(df_e_clumps, num_res, box_dimensions)
# display(df_e_centroids)

e_centroids = {res_id + 1: e_centroid for res_id, e_centroid in enumerate(df_e_centroids.values)}
e_centroids[100]

array([101.35818182,  98.56590909,  58.96045455])

### Visualisation

In [30]:
# e_centroid_100 = df_e_centroids.loc[100].to_numpy()
# e_mesh = trimesh.creation.box(extents=[1.0, 1.0, 1.0])
# e_mesh.apply_translation(e_centroid_100)

# mesh_100 = mol_meshes[100]
# mesh_100

# scene = trimesh.Scene([mesh_100, e_mesh])
# scene.show()

## Nearest Neighbour Candidate

For this part, the true centroid of molecule is calculated  as`centroids` because it is needed for `radii`.

The function of `radii` is to help with blocking algorithm.

In [15]:
def compute_centroids_and_radii_pbc(mol_meshes, box_dimensions):
    """
    Compute periodic-boundary-condition (PBC) aware centroids and radii for a set of molecules.

    For each molecule mesh, this function:
      1. Selects a reference vertex (atom) as the origin.
      2. Unwraps all other vertices relative to this reference using the minimum-image convention (MIC),
         so that all atoms are locally unwrapped and contiguous in space.
      3. Computes the centroid (geometric center) of the unwrapped coordinates.
      4. Calculates the maximum MIC distance from the centroid to any vertex, defining the molecule's effective radius.

    Parameters
    ----------
    mol_meshes : dict[int, trimesh.Trimesh]
        Dictionary mapping molecule IDs to their corresponding trimesh mesh objects.
    box_dimensions : array-like, shape (3,) or (3,3)
        Simulation box dimensions (in Å). Should be a 3-element array for orthorhombic boxes.

    Returns
    -------
    centroids : dict[int, np.ndarray]
        Dictionary mapping molecule IDs to their centroid coordinates (in Å), unwrapped in PBC.
    radii : dict[int, float]
        Dictionary mapping molecule IDs to their effective radii (in Å), defined as the maximum MIC distance
        from the centroid to any vertex in the molecule.
    """
    centroids, radii = {}, {}
    # box_dimensions = np.array(box_dimensions, dtype=float)

    for mol_id, mesh in mol_meshes.items():
        verts = mesh.vertices
        ref = verts[0]  # reference atom
        disp = mic_vector(verts - ref, box_dimensions)
        unwrapped = ref + disp

        # centroid in unwrapped space
        center = unwrapped.mean(axis=0)

        # MIC distances from centroid to each vertex
        disp_center = mic_vector(verts - center, box_dimensions)
        radius = np.linalg.norm(disp_center, axis=1).max()

        centroids[mol_id] = center
        radii[mol_id] = radius

    return centroids, radii

def nearest_neighbors(mol_meshes, box, centroids, k=10, return_meshes=False):
    """
    Find the k nearest neighbors of each molecule, based on centroid distance.
    
    Parameters
    ----------
    mol_meshes : dict[int, trimesh.Trimesh]
        Dictionary of molecule meshes.
    box : float or array-like
        Simulation box length (for PBC). Pass a scalar if cubic.
    k : int
        Number of nearest neighbors to return per molecule.

    Returns
    -------
    dict[int, list[tuple[int,float]]]
        Mapping mol_id -> list of (neighbor_id, distance).
    """

    ids = list(centroids.keys())
    coords = np.vstack([centroids[i] for i in ids])
    coords_wrapped = wrap_points(coords, box)
    kd = KDTree(coords_wrapped, boxsize=box)

    neighbors = {}
    for idx, mol_id in enumerate(ids):
        dists, idxs = kd.query(coords[idx], k=k+1)
        dists, idxs = dists[1:], idxs[1:]
        if return_meshes:
            neighbors[mol_id] = [(mol_meshes[ids[j]], float(d)) for j, d in zip(idxs, dists)]
        else:
            neighbors[mol_id] = [(ids[j], float(d)) for j, d in zip(idxs, dists)]
    return neighbors


In [16]:
centroids, radii = compute_centroids_and_radii_pbc(mol_meshes, box_dimensions)

neighbors = nearest_neighbors(mol_meshes, box_dimensions, centroids, k=10, return_meshes=False)

neighbor_candidates = [(key, t[0]) for key, value in neighbors.items() for t in value]

neighbor_candidates_sorted = [(min(key, t[0]), max(key, t[0])) 
                      for key, value in neighbors.items() for t in value if key < t[0]]

neighbor_candidates_sorted = list(set(neighbor_candidates_sorted))
neighbor_candidates_sorted.sort()

In [6]:
# print(centroids)
# print(radii)
# print(neighbors)
# print(neighbor_candidates_sorted)

## Blocking Algorithm

No parallelization for now, uses too much memory

In [ ]:
def blocked_by_any(i, j, e_centroids, centroids, radii, mol_meshes):
    """
    Determine if the direct path between two molecule e_centroids is obstructed by any other molecule.

    For a given pair of molecules (i, j), this function checks whether the straight line
    connecting their e_centroids is intersected ("blocked") by any other molecule in the system.
    The check is performed in two steps:
      1. Fast sphere rejection: For each candidate blocking molecule, if its centroid is not
         within its effective radius of the line segment, it is skipped.
      2. Ray-mesh intersection: If the sphere check passes, a ray-mesh intersection test is
         performed to determine if the mesh of the candidate molecule blocks the path.

    Periodic boundary conditions (PBC) are handled using the minimum-image convention.

    Parameters
    ----------
    i, j : int
        IDs of the two molecules to test for a direct connection.
    e_centroids : dict[int, np.ndarray]
        Dictionary mapping molecule IDs to their centroid coordinates (in Å).
    radii : dict[int, float]
        Dictionary mapping molecule IDs to their effective radii (in Å).
    mol_meshes : dict[int, trimesh.Trimesh]
        Dictionary mapping molecule IDs to their trimesh mesh objects.

    Returns
    -------
    blocked : bool
        True if the path between i and j is blocked by any other molecule, False otherwise.
    """
    # Map molecule IDs to their index in ids

    ci, cj = e_centroids[i], e_centroids[j]
    seg_vec = cj - ci
    seg_len = np.linalg.norm(seg_vec)
    if seg_len < 1e-6:
        return False
    direction = seg_vec / seg_len

    # Get candidate molecule IDs (not indices)
    cand_ids = [t[1] for t in neighbor_candidates if t[0] == i]

    for mol_k in cand_ids:
        if mol_k in (i, j):
            continue

        # Quick sphere reject
        ck = centroids[mol_k]
        v = cj - ci
        w = ck - ci
        proj = np.dot(w, v) / np.dot(v, v)
        proj = np.clip(proj, 0.0, 1.0)
        closest = ci + proj * v
        if np.linalg.norm(ck - closest) > radii[mol_k]:
            continue

        # Expensive ray test
        if mol_meshes[mol_k].ray.intersects_any(
            ray_origins=ci.reshape(1, 3),
            ray_directions=direction.reshape(1, 3)
        ):
            return True
    return False

In [ ]:
def find_neighbors(e_centroids, centroids, radii, mol_meshes, neighbor_candidates):
    neighbor_pairs = []
    """
    Determine all unblocked neighbor pairs from a list of candidate molecule pairs.

    Parameters
    ----------
    centroids : dict[int, np.ndarray]
        Dictionary mapping molecule IDs to their centroid coordinates (in Å).
    radii : dict[int, float]
        Dictionary mapping molecule IDs to their effective radii (in Å).
    mol_meshes : dict[int, trimesh.Trimesh]
        Dictionary mapping molecule IDs to their trimesh mesh objects.
    neighbor_candidates : list[tuple[int, int]]
        List of candidate neighbor pairs (i, j) to test for blocking.

    Returns
    -------
    neighbor_pairs : list[tuple[int, int]]
        List of unblocked neighbor pairs (i, j).
    """

    ids = list(centroids.keys())

    neighbor_pairs = []
    for pair in tqdm(neighbor_candidates, desc="Testing neighbor pairs"):
        i, j = pair
        if not blocked_by_any(i, j, e_centroids, centroids, radii, mol_meshes):
            neighbor_pairs.append((i, j))
            
    return neighbor_pairs

In [ ]:
results_2 = find_neighbors(e_centroids, centroids, radii, mol_meshes, box_dimensions, neighbor_candidates_sorted)
print("Number of unblocked pairs:",len(results_2))

Testing neighbor pairs:  18%|█▊        | 1321/7511 [02:45<6:12:57,  3.62s/it]

## Save the result

In [ ]:
# Save to csv
with open("../../data/processed/unblocked_pairs_1.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerows(results)

# with open("../../data/processed/unblocked_pairs_2.csv", "w", newline="") as f:
#     writer = csv.writer(f)
#     writer.writerows(results_2)

In [ ]:
# Load back
with open("../../data/processed/unblocked_pairs_1.csv") as f:
    reader = csv.reader(f)
    results_1 = [tuple(map(int, row)) for row in reader]

with open("../../data/processed/unblocked_pairs_2.csv") as f:
    reader = csv.reader(f)
    results_2 = [tuple(map(int, row)) for row in reader]

In [9]:
def neighbor_ids(i):
    l1 = [t[1] for t in results_1 if t[0] == i]
    l2 = [t[1] for t in results_2 if t[0] == i]
    print(l1,l2)

    return l1, l2

In [10]:
l1, l2 = neighbor_ids(290)
submesh = [mol_meshes[nid] for nid in l2]
scene = trimesh.Scene(submesh)
scene.show()

NameError: name 'results_1' is not defined